# Lab 13: GLM, ROC Curve & Full Classification Pipeline
> Week 13 | CLO3 | ISLP Ch.4.6–4.7

## บทนำสัปดาห์

สัปดาห์นี้เราจะเรียนรู้ 3 สิ่งสำคัญที่รวม classification knowledge จาก Week 11–12 เข้าด้วยกัน ส่วนแรกคือ **Generalized Linear Models (GLM)** ซึ่งเป็น unifying framework ที่รวม Linear Regression, Logistic Regression และ Poisson Regression ไว้ใน family เดียวกัน ส่วนที่สองคือ **ROC Curve และ AUC** ซึ่งเป็นเครื่องมือมาตรฐานในการเปรียบเทียบ classifier โดยไม่ขึ้นกับ threshold ส่วนที่สามคือ **Full Classification Pipeline** ที่รวมทุกขั้นตอนตั้งแต่ preprocessing จนถึงการ select best model **เป้าหมาย** คือนักศึกษาสามารถ fit Poisson Regression สำหรับ count data, สร้าง ROC curve เปรียบเทียบหลาย classifier และสร้าง end-to-end classification pipeline ที่ professional ได้ ทักษะเหล่านี้ใช้ใน data science project จริงทุกโครงการ ตั้งแต่ fraud detection, medical diagnosis ไปจนถึง demand forecasting

## สิ่งที่จะเรียนรู้
- GLM framework: link function, Poisson regression สำหรับ count data
- ROC Curve: construction, AUC interpretation
- Full pipeline: preprocessing → train → evaluate → compare → select
- Class imbalance handling: threshold, class_weight

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ครบสำหรับ GLM, ROC และ full pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

# ─── Classifiers ───────────────────────────────────────────────────────────────
# วัตถุประสงค์: import classifiers ทั้งหมดที่จะเปรียบเทียบใน full pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score, classification_report
)
from scipy.special import expit

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Setup complete ✓')

## Part 1: Generalized Linear Models (GLM)

**Part นี้เราจะเรียนรู้ GLM framework** ซึ่งเป็น generalization ของ Linear Regression ไปสู่ response Y ที่มี distribution ได้หลายแบบ แนวคิดหลักคือ **link function** g(μ) ที่เชื่อม mean ของ Y กับ linear predictor β₀ + Σβⱼxⱼ เราจะ focus ที่ **Poisson Regression** สำหรับ count data เช่น จำนวนจักรยาน, จำนวนครั้งที่โทรหา customer service, จำนวน insects ที่จับได้

In [ ]:
# ─── สร้าง Bikeshare-like dataset ──────────────────────────────────────────────
# วัตถุประสงค์: simulate bike rental data ที่มี count response
# ใช้ Poisson model จริง: log(λ) = 3 + 0.3*hour_normalized - 0.5*rain
np.random.seed(42)
n = 1000

# สร้าง features
hour     = np.random.randint(0, 24, n)       # ชั่วโมง (0–23)
temp     = np.random.uniform(5, 35, n)       # อุณหภูมิ (°C)
rain     = np.random.binomial(1, 0.2, n)     # ฝนตก (0/1)
weekday  = np.random.binomial(1, 5/7, n)     # วันธรรมดา (0/1)

# True Poisson model: log(λ) = linear predictor
hour_normalized = (hour - 12) / 12  # normalize to [-1, 1]
log_lambda = (3.5
              + 0.5 * hour_normalized      # ช่วงกลางวันมีคนใช้มาก
              - 0.3 * np.abs(hour_normalized)  # peak ตรงกลาง
              + 0.05 * temp                # อุณหภูมิดีคนออกมาปั่น
              - 1.2 * rain                 # ฝนตกคนไม่ปั่น
              + 0.3 * weekday)             # วันธรรมดาคนใช้มาก
count = np.random.poisson(np.exp(log_lambda))   # Poisson samples

bike = pd.DataFrame({
    'count': count, 'hour': hour, 'temp': temp,
    'rain': rain, 'weekday': weekday,
    'hour_normalized': hour_normalized
})

print(f'Bikeshare dataset: {bike.shape}')
print(f'Count stats: mean={count.mean():.1f}, max={count.max()}, min={count.min()}')
bike.head()

In [ ]:
# ─── EDA: Distribution ของ count ──────────────────────────────────────────────
# วัตถุประสงค์: แสดงว่า count data ไม่ได้ Normal → ต้องใช้ Poisson

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram ของ count
axes[0].hist(bike['count'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Bike Count'); axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Count (right-skewed)')

# Count by hour
axes[1].scatter(bike['hour'], bike['count'], alpha=0.1, color='steelblue', s=10)
hourly_mean = bike.groupby('hour')['count'].mean()
axes[1].plot(hourly_mean.index, hourly_mean.values, 'r-', lw=2, label='Mean')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('Count')
axes[1].set_title('Count vs Hour')
axes[1].legend()

# Boxplot: Rain vs No Rain
bike.boxplot(column='count', by='rain', ax=axes[2])
axes[2].set_title('Count by Rain (0=No, 1=Yes)')
axes[2].set_xlabel('Rain'); axes[2].set_ylabel('Count')
plt.suptitle('')

plt.tight_layout()
plt.show()
print('สังเกต: count มี right-skewed distribution — Poisson เหมาะกว่า Normal!')

In [ ]:
# ─── Fit Poisson Regression ────────────────────────────────────────────────────
# วัตถุประสงค์: สร้าง Poisson GLM ด้วย statsmodels
# Link function: log → log(λ) = β₀ + β₁×hour + β₂×temp + β₃×rain + β₄×weekday

formula = 'count ~ hour_normalized + temp + rain + weekday'

# Fit Poisson GLM
poisson_model = smf.glm(
    formula    = formula,
    data       = bike,
    family     = sm.families.Poisson()  # Poisson family + log link
).fit()

print(poisson_model.summary())

### 🔰 TODO 1 (Easy): ตีความ Poisson Regression Coefficients

Poisson Regression ใช้ **log link** หมายความว่า β̂ ไม่ได้บอก change ของ count โดยตรง แต่บอก change ของ **log(λ)** การตีความที่ถูกต้องคือ: e^β̂ = **Incidence Rate Ratio (IRR)** ซึ่งบอกว่า λ (mean count) เปลี่ยนแปลงเท่าไร เมื่อ X เพิ่ม 1 หน่วย

**สิ่งที่ต้องทำ:**
1. อ่านค่า β̂ ของแต่ละตัวแปรจาก summary
2. คำนวณ IRR = e^β̂ สำหรับทุก coefficient
3. สร้าง DataFrame แสดง: coef, IRR, p-value
4. ตีความ β̂_rain และ β̂_temp ในประโยค (1–2 ประโยค)
5. Compare: Poisson vs Linear Regression บนข้อมูลเดียวกัน — plot prediction ทั้งคู่

**Expected**: rain coefficient ≈ -1.2 → IRR ≈ 0.30 → ฝนตกลด bike rental 70%

In [ ]:
# TODO 1: ตีความ Poisson Regression Coefficients
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. คำนวณ IRR = e^β̂


# 2. สร้าง DataFrame


# 3. Compare Poisson vs Linear
# Fit Linear Regression


# Plot predictions


# 4. คำอธิบาย:
print('β̂_rain =', ___)
print('IRR_rain = e^β̂_rain =', ___)
print('ความหมาย: ...')


## Part 2: ROC Curve และ AUC

**Part นี้เราจะสร้าง ROC Curve** ซึ่งเป็นเครื่องมือสำคัญในการเปรียบเทียบ classifier ROC Curve แสดง trade-off ระหว่าง True Positive Rate (Sensitivity/Recall) และ False Positive Rate (1−Specificity) ที่ทุก possible threshold AUC (Area Under Curve) สรุป ROC curve เป็น scalar เดียว — classifier ที่ดีมี AUC ใกล้ 1.0 ข้อดีของ AUC คือ **threshold-independent** — เราเปรียบเทียบ classifier ได้โดยไม่ต้องเลือก threshold ก่อน

In [ ]:
# ─── โหลด Default dataset ──────────────────────────────────────────────────────
# วัตถุประสงค์: ใช้ Default dataset เป็น dataset สำหรับ ROC comparison
try:
    default = pd.read_csv('https://www.statlearning.com/s/Default.csv', index_col=0)
    default.columns = default.columns.str.lower()
except:
    np.random.seed(0); n = 10000
    balance = np.clip(np.random.exponential(900, n), 0, 3000)
    income  = np.random.normal(35000, 15000, n)
    student = np.random.choice([0, 1], n, p=[0.7, 0.3])
    log_odds = -10.65 + 0.0055*balance - 0.000002*income - 0.65*student
    default_y = np.random.binomial(1, expit(log_odds))
    default = pd.DataFrame({
        'default': ['Yes' if d else 'No' for d in default_y],
        'student': ['Yes' if s else 'No' for s in student],
        'balance': balance, 'income': income
    })

default['default_num'] = (default['default'] == 'Yes').astype(int)
default['student_num'] = (default['student'] == 'Yes').astype(int)

X = default[['balance', 'income', 'student_num']]
y = default['default_num']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print('Default dataset ready ✓')

In [ ]:
# ─── สร้าง ROC Curve: Logistic vs LDA ─────────────────────────────────────────
# วัตถุประสงค์: plot ROC Curve ของ 2 classifiers บน Default test set
# ROC Curve = (FPR, TPR) ที่ threshold ต่างๆ ตั้งแต่ 0 ถึง 1

# Fit Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_proba = lr.predict_proba(X_test)[:, 1]

# Fit LDA
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
lda_proba = lda.predict_proba(X_test)[:, 1]

# คำนวณ ROC curve
fpr_lr, tpr_lr, thresh_lr = roc_curve(y_test, lr_proba)
fpr_lda, tpr_lda, thresh_lda = roc_curve(y_test, lda_proba)

auc_lr  = roc_auc_score(y_test, lr_proba)
auc_lda = roc_auc_score(y_test, lda_proba)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
axes[0].plot(fpr_lr,  tpr_lr,  'steelblue', lw=2.5, label=f'Logistic (AUC={auc_lr:.3f})')
axes[0].plot(fpr_lda, tpr_lda, 'tomato',    lw=2.5, label=f'LDA (AUC={auc_lda:.3f})',
             linestyle='--')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
axes[0].set_title('ROC Curve: Logistic vs LDA')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Trade-off ที่ different thresholds
thresholds_to_show = [0.1, 0.2, 0.3, 0.5, 0.8]
precision_vals, recall_vals = [], []
for t in np.linspace(0.05, 0.95, 50):
    pred_t = (lr_proba >= t).astype(int)
    precision_vals.append(precision_score(y_test, pred_t, zero_division=0))
    recall_vals.append(recall_score(y_test, pred_t, zero_division=0))

axes[1].plot(recall_vals, precision_vals, 'purple', lw=2)
axes[1].set_xlabel('Recall (Sensitivity)', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curve (Logistic Regression)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Logistic AUC: {auc_lr:.4f}')
print(f'LDA AUC:      {auc_lda:.4f}')
print('ทั้ง 2 method ให้ AUC ใกล้เคียงกันมาก — LDA ทำงานดีเท่า Logistic บน Default!')

### 🔰 TODO 2 (Medium): ROC + Threshold Selection

ROC Curve บอก trade-off ระหว่าง sensitivity และ specificity แต่เราต้องเลือก threshold 1 ค่าในที่สุด สำหรับปัญหาที่ **FN มีราคาสูงกว่า FP** (เช่น การ miss default ของธนาคาร) ควรเลือก threshold ต่ำกว่า 0.5 เพื่อเพิ่ม recall ใน TODO นี้คุณจะหา **optimal threshold** โดยใช้ Youden's J statistic และ F-beta score

**สิ่งที่ต้องทำ:**
1. คำนวณ **Youden's J statistic** = TPR − FPR สำหรับทุก threshold บน ROC curve ของ Logistic
2. หา optimal threshold ที่ J สูงสุด
3. เปรียบเทียบ metrics ที่ threshold = 0.5 vs optimal threshold:

| Threshold | Precision | Recall | F1 | Accuracy |
|-----------|-----------|--------|----|----------|
| 0.5 | | | | |
| Optimal (Youden) | | | | |

4. Plot ROC curve พร้อมจุด optimal threshold (ใช้ scatter marker บน curve)

**Hint**: `fpr_lr, tpr_lr, thresh_lr = roc_curve(y_test, lr_proba)` แล้ว `J = tpr_lr - fpr_lr`

In [ ]:
# TODO 2: ROC + Optimal Threshold (Youden's J)
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# 1. คำนวณ Youden's J = TPR - FPR


# 2. หา optimal threshold


# 3. Comparison table


# 4. Plot ROC curve + optimal point


## Part 3: Full Classification Pipeline

**Part นี้เราจะสร้าง full classification pipeline บน Stock Market (Smarket) dataset** ตาม ISLP 4.7.1 pipeline นี้ครอบคลุมทุกขั้นตอน: (1) load + explore, (2) preprocess, (3) fit ทุก classifiers, (4) evaluate, (5) select best model Pipeline ที่ดีต้องทำซ้ำได้ เปรียบเทียบได้ และอธิบายได้ว่าทำไมถึงเลือก model นั้น

In [ ]:
# ─── โหลด Smarket dataset ──────────────────────────────────────────────────────
# วัตถุประสงค์: Smarket = S&P500 2001-2005, predict Direction Up/Down
try:
    smarket = pd.read_csv('https://www.statlearning.com/s/Smarket.csv', index_col=0)
    smarket.columns = smarket.columns.str.lower()
except:
    np.random.seed(1); n = 1250
    lags = np.random.normal(0, 1, (n, 5))
    volume = np.random.uniform(1, 2, n)
    today  = np.random.normal(0, 1, n)
    direction = np.where(today > 0, 'Up', 'Down')
    smarket = pd.DataFrame(lags, columns=[f'lag{i+1}' for i in range(5)])
    smarket['volume'] = volume
    smarket['today'] = today
    smarket['direction'] = direction
    smarket['year'] = np.repeat(range(2001, 2006), 250)

smarket['direction_num'] = (smarket['direction'] == 'Up').astype(int)

# ─── Time-based train/test split ──────────────────────────────────────────────
# วัตถุประสงค์: ใช้ time-based split ป้องกัน look-ahead bias ใน time series
features = ['lag1', 'lag2', 'lag3', 'lag4', 'lag5', 'volume']
train_mask = smarket['year'] <= 2004
test_mask  = smarket['year'] == 2005

X_sm_tr = smarket.loc[train_mask, features]
y_sm_tr = smarket.loc[train_mask, 'direction_num']
X_sm_te = smarket.loc[test_mask, features]
y_sm_te = smarket.loc[test_mask, 'direction_num']

print(f'Train: {X_sm_tr.shape[0]} obs (2001-2004)')
print(f'Test:  {X_sm_te.shape[0]} obs (2005)')
print(f'Class balance (test): {y_sm_te.value_counts().to_dict()}')

In [ ]:
# ─── Standardize features ──────────────────────────────────────────────────────
# วัตถุประสงค์: KNN และ Logistic Regression ต้องการ scaled features
# CRITICAL: fit scaler บน train เท่านั้น ไม่ใช้ test ข้อมูลใน fit!

scaler = StandardScaler()
X_sm_tr_sc = scaler.fit_transform(X_sm_tr)  # fit + transform train
X_sm_te_sc = scaler.transform(X_sm_te)      # transform test เท่านั้น

print('Feature scaling complete ✓')
print(f'Train mean after scaling: {X_sm_tr_sc.mean(axis=0).round(4)}')  # ≈ 0
print(f'Train std after scaling:  {X_sm_tr_sc.std(axis=0).round(4)}')   # ≈ 1

In [ ]:
# ─── Full Pipeline: Fit All Classifiers ───────────────────────────────────────
# วัตถุประสงค์: fit และ evaluate classifier ทุกตัวใน loop เดียว
# ทำให้ code สั้นและ scalable — เพิ่ม classifier ใหม่แค่เพิ่ม dict entry

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'LDA'                : LinearDiscriminantAnalysis(),
    'QDA'                : QuadraticDiscriminantAnalysis(),
    'Naive Bayes'        : GaussianNB(),
    'KNN (K=10)'         : KNeighborsClassifier(n_neighbors=10)
}

results = []
for name, clf in classifiers.items():
    # Fit บน train
    clf.fit(X_sm_tr_sc, y_sm_tr)
    
    # Predict บน test
    y_pred  = clf.predict(X_sm_te_sc)
    y_proba = clf.predict_proba(X_sm_te_sc)[:, 1]
    
    # คำนวณ metrics ทั้งหมด
    results.append({
        'Method'    : name,
        'Accuracy'  : round(accuracy_score(y_sm_te, y_pred), 4),
        'Precision' : round(precision_score(y_sm_te, y_pred, zero_division=0), 4),
        'Recall'    : round(recall_score(y_sm_te, y_pred, zero_division=0), 4),
        'F1'        : round(f1_score(y_sm_te, y_pred, zero_division=0), 4),
        'AUC'       : round(roc_auc_score(y_sm_te, y_proba), 4)
    })

results_df = pd.DataFrame(results)
print('Full Pipeline Results — Smarket 2005 Test Set:')
print(results_df.to_string(index=False))

In [ ]:
# ─── ROC Curves: All 5 Classifiers ────────────────────────────────────────────
# วัตถุประสงค์: plot ROC ทุก method ในกราฟเดียวเพื่อ visual comparison

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['steelblue', 'tomato', 'green', 'purple', 'orange']

# ROC Curves
for (name, clf), color in zip(classifiers.items(), colors):
    y_proba = clf.predict_proba(X_sm_te_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_sm_te, y_proba)
    auc = roc_auc_score(y_sm_te, y_proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} ({auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Smarket 2005')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Accuracy Comparison Bar Chart
results_df.plot(x='Method', y=['Accuracy', 'F1', 'AUC'],
                kind='bar', ax=axes[1], color=['steelblue', 'tomato', 'green'])
axes[1].set_title('Metrics Comparison — Smarket 2005')
axes[1].set_xlabel('')
axes[1].set_ylabel('Score')
axes[1].set_xticklabels(results_df['Method'], rotation=20, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

### 🔰 TODO 3 (Medium): Class Imbalance Handling

ปัญหา class imbalance เป็นเรื่องปกติใน real-world data เช่น fraud detection (1% fraud), rare disease (2% positive) classifier ที่ predict majority class เสมอจะได้ accuracy สูง แต่ recall ต่ำ วิธีแก้ที่ง่ายที่สุดคือ `class_weight='balanced'` ซึ่งให้ weight แก่ minority class มากขึ้น ใน TODO นี้คุณจะสร้าง imbalanced dataset เองและทดสอบ effect ของ class_weight

**สิ่งที่ต้องทำ:**
1. สร้าง imbalanced dataset ด้วย `make_classification(weights=[0.95, 0.05])`
2. Fit Logistic Regression 2 versions: standard และ `class_weight='balanced'`
3. เปรียบเทียบ confusion matrix ของทั้ง 2 versions
4. สรุป: `class_weight='balanced'` ช่วย metric ไหน? เสียสละ metric ไหน?

**Hint**: `from sklearn.datasets import make_classification`

In [ ]:
# TODO 3: Class Imbalance Handling
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────
from sklearn.datasets import make_classification

# 1. สร้าง imbalanced dataset
Ximb, yimb = make_classification(
    n_samples=2000, n_features=10, n_informative=5,
    weights=[0.95, 0.05],  # 95% class 0, 5% class 1
    random_state=42
)
# X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = ...


# 2. Fit 2 versions


# 3. Confusion matrices + metrics


# 4. สรุป:
print('class_weight=balanced ช่วย: ...')
print('เสียสละ: ...')


### 🔰 TODO 4 (Hard): Full Pipeline บน Real Dataset

ถึงเวลาทดสอบทักษะทั้งหมด คุณจะสร้าง **full end-to-end pipeline** บน **Titanic-like dataset** จาก sklearn เป้าหมายคือ predict การรอดชีวิต (survived=1) จาก passenger features นี่เป็น classic binary classification ที่มี class imbalance เล็กน้อยและ features ผสมกัน (continuous + categorical)

**สิ่งที่ต้องทำ:**
1. โหลด dataset ด้วย synthetic Titanic data (code ให้แล้ว)
2. EDA: แสดง survival rate by class, sex, age group
3. Preprocess: encode categorical, handle missing, scale
4. Fit ทุก 5 classifiers พร้อม StandardScaler
5. Full evaluation: Accuracy, Precision, Recall, F1, AUC
6. Plot ROC Curve ของทุก 5 classifiers
7. เลือก best model + อธิบาย: ทำไม? ใช้ metric ไหนในการตัดสินใจ?
8. ปรับ threshold ของ best model ให้ Recall ≥ 0.80 — trade-off Precision เท่าไร?

**Note**: งาน Real World มักไม่มี "perfect" dataset — การจัดการ preprocessing อย่างถูกต้องสำคัญมาก

In [ ]:
# TODO 4: Full Pipeline — Titanic-like Dataset
# ─── TODO: เขียน code ที่นี่ ──────────────────────────────────────────────────

# สร้าง synthetic Titanic dataset (code ให้แล้ว)
np.random.seed(0)
n_titanic = 891
pclass  = np.random.choice([1, 2, 3], n_titanic, p=[0.24, 0.21, 0.55])
sex_num = np.random.binomial(1, 0.35, n_titanic)  # 1=female
age     = np.clip(np.random.normal(30, 14, n_titanic), 0.5, 80)
fare    = np.random.exponential(30, n_titanic)
sibsp   = np.random.choice(range(9), n_titanic, p=[0.68,0.23,0.03,0.02,0.01,0.01,0.01,0.005,0.005])

# Survival probability based on known patterns
log_odds_surv = (-1.0
                 + 1.5 * sex_num          # women survive more
                 - 0.4 * (pclass == 3)    # 3rd class lower survival
                 + 0.2 * (pclass == 1)    # 1st class higher
                 - 0.01 * age             # older → lower survival
                 + 0.003 * fare)          # higher fare → higher survival
survived = np.random.binomial(1, expit(log_odds_surv))

titanic = pd.DataFrame({
    'survived': survived, 'pclass': pclass, 'sex_num': sex_num,
    'age': age, 'fare': fare, 'sibsp': sibsp
})
print(f'Titanic dataset: {titanic.shape}')
print(f'Survival rate: {survived.mean():.2%}')

# 1. EDA


# 2+3. Split + Preprocess


# 4+5. Fit + Evaluate


# 6. ROC Curve


# 7. Best model recommendation
print('Best model: ...')

# 8. Threshold tuning


## Case Study: Poisson Regression สำหรับ Customer Service Call Volume

**Scenario**  
ธนาคารต้องการ forecast จำนวน incoming calls ไปยัง call center ในแต่ละชั่วโมง เพื่อ optimize staffing บางชั่วโมง call สูงมาก (lunch break, evening) บางชั่วโมงต่ำ ถ้าเอา Linear Regression มาใช้อาจ predict negative calls ได้ ซึ่งไม่สมเหตุสมผล

**Data**: 30 วัน × 24 ชั่วโมง = 720 observations, features: hour, weekday, is_salary_day

**Method — Poisson Regression:**  
log(λ) = β₀ + β₁×hour + β₂×weekday + β₃×salary_day  
→ λ̂ = จำนวน calls ที่คาดหวัง (เสมอ > 0)

**Result**  
- β̂_salary_day = 0.85 → e^0.85 ≈ 2.3 → salary day มี calls มากกว่า 2.3 เท่า  
- RMSE Poisson: 12.3 vs Linear: 14.7 (Poisson ดีกว่า 16%)

**Insight**  
Poisson Regression ให้ predictions ที่สมเหตุสมผลกว่า (ไม่ negative) และ RMSE ต่ำกว่า ใช้ forecast staffing ได้โดยตรง

## สรุปสิ่งที่เรียนรู้

| Concept | ใช้เมื่อ | Python |
|---------|---------|--------|
| Poisson GLM | Y เป็น count (≥0) | `smf.glm(..., family=sm.families.Poisson())` |
| ROC Curve | เปรียบเทียบ classifier | `roc_curve()`, `roc_auc_score()` |
| AUC | Threshold-free comparison | AUC ∈ [0,1], >0.8 = ดี |
| Youden's J | หา optimal threshold | J = TPR − FPR, max J |
| class_weight='balanced' | Imbalanced data | sklearn classifiers parameter |
| StandardScaler | Before KNN, LR | `fit` บน train เท่านั้น! |

## คำถาม Reflection

1. **AUC vs Accuracy**: สถานการณ์ใดที่ AUC สูงแต่ accuracy ต่ำ? เป็นไปได้ไหม? ถ้าได้จะเกิดขึ้นได้อย่างไร?

2. **GLM vs Transformation**: แทนที่จะใช้ Poisson Regression เราสามารถ transform Y ด้วย log แล้วใช้ Linear Regression ได้ไหม? วิธีนี้มีข้อดีข้อเสียอะไรเมื่อเทียบกับ Poisson GLM?